<a href="https://colab.research.google.com/github/4232501030-Ghaby/Pemograman-Dasar/blob/main/TUGAS_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# NAMA : GHABY PATTAMA SINAGA
# NIM : 4232501030
# TUGAS 2: Modul KPI Pembangkit

# DATA Unit 1, 1-30 November 2026 (hari ke-1 di indeks 0)
energi = [5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
          6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
          5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062]   # MWh
batubara = [2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
            3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
            3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240]  # ton

jam_operasi = [24] * 30
jam_operasi[16] = 18   # hari ke-17 hanya beroperasi 18 jam


# FUNGSI
def heat_rate(batubara_t, energi_mwh, nilai_kalor_kcal_kg=4400):
    """Heat rate (kcal/kWh) dari batubara (t) dan energi terkirim (MWh)."""
    if energi_mwh <= 0:
        raise ValueError("energi_mwh harus positif")
    return batubara_t * nilai_kalor_kcal_kg / energi_mwh


def efisiensi_termal(heat_rate_kcal_kwh):
    """Efisiensi termal (%) dari heat rate. 1 kWh = 860 kcal."""
    return 860 / heat_rate_kcal_kwh * 100


def capacity_factor(energi_mwh, daya_mw=330, jam=24):
    """Capacity factor (%) = energi / (daya terpasang x jam) x 100."""
    return energi_mwh / (daya_mw * jam) * 100


def konsumsi_spesifik(batubara_t, energi_mwh):
    """Konsumsi batubara spesifik (t/MWh)."""
    return batubara_t / energi_mwh


def status_hari(efisiensi, jam_operasi):
    """'TIDAK PENUH' jika jam operasi < 24, 'EFISIENSI RENDAH' jika
    efisiensi < 36,3 %, selain itu 'NORMAL'."""
    if jam_operasi < 24:
        return "TIDAK PENUH"
    if efisiensi < 36.3:
        return "EFISIENSI RENDAH"
    return "NORMAL"


#  UJI assert
assert heat_rate(1, 2) == 2200                       # 1 x 4400 / 2
assert round(efisiensi_termal(2150), 2) == 40        # 860 / 2150 = 0,4
assert round(capacity_factor(7920)) == 100           # 330 x 24 = 7920
assert konsumsi_spesifik(100, 200) == 0.5            # 100 t / 200 MWh
assert status_hari(40, 18) == "TIDAK PENUH"
assert status_hari(36.0, 24) == "EFISIENSI RENDAH"
assert status_hari(36.3, 24) == "NORMAL"
print("Semua uji lulus")


#  FUNGSI PENGHITUNG & PROSEDUR CETAK
def hitung_harian(energi, batubara, jam_operasi, daya_mw, nilai_kalor):
    """Hitung KPI tiap hari; kembalikan daftar (hari, hr, efi, cf, scc, status)."""
    hasil = []
    for i in range(len(energi)):
        hr = heat_rate(batubara[i], energi[i], nilai_kalor)
        efi = efisiensi_termal(hr)
        cf = capacity_factor(energi[i], daya_mw, jam_operasi[i])
        scc = konsumsi_spesifik(batubara[i], energi[i])
        status = status_hari(efi, jam_operasi[i])
        hasil.append((i + 1, hr, efi, cf, scc, status))
    return hasil


def cetak_laporan(hasil):
    """Prosedur: cetak tabel KPI harian (hari, HR, efisiensi, CF, SCC, status)."""
    print(f"{'Hari':>4} {'HR kcal/kWh':>12} {'Efi %':>7} {'CF %':>6} "
          f"{'SCC t/MWh':>10}  Status")
    for hari, hr, efi, cf, scc, status in hasil:
        print(f"{hari:>4} {hr:>12,.0f} {efi:>7.2f} {cf:>6.1f} "
              f"{scc:>10.3f}  {status}")


def cetak_ringkasan(hasil, energi, jam_operasi, daya_mw):
    """Prosedur: cetak ringkasan bulanan."""
    efisiensi = [h[2] for h in hasil]
    jumlah_status = {}
    for h in hasil:
        jumlah_status[h[5]] = jumlah_status.get(h[5], 0) + 1

    cf_bulanan = capacity_factor(sum(energi), daya_mw, sum(jam_operasi))
    terburuk = sorted(hasil, key=lambda h: h[2])[:3]

    print(f"\nRata-rata efisiensi : {sum(efisiensi) / len(efisiensi):.2f} %")
    print(f"Capacity factor     : {cf_bulanan:.1f} %")
    print("Jumlah hari per status:")
    for status, jumlah in jumlah_status.items():
        print(f"  {status:<17}: {jumlah} hari")
    print("Tiga hari paling tidak efisien:")
    for h in terburuk:
        print(f"  Hari {h[0]:>2} : {h[2]:.2f} % ({h[5]})")


# PROGRAM UTAMA: Unit 1 (330 MW, 4400 kcal/kg), 30 hari
print("\n=== LAPORAN UNIT 1 ===")
hasil_unit1 = hitung_harian(energi, batubara, jam_operasi, 330, 4400)
cetak_laporan(hasil_unit1)
cetak_ringkasan(hasil_unit1, energi, jam_operasi, 330)


# PAKAI ULANG: Unit 2 (600 MW, 5100 kcal/kg), 5 hari
energi_u2 = [12150, 12080, 11020, 12210, 12095]
batubara_u2 = [5240, 5225, 4790, 5260, 5230]
jam_u2 = [24] * 5

print("\n=== LAPORAN UNIT 2 ===")
hasil_unit2 = hitung_harian(energi_u2, batubara_u2, jam_u2, 600, 5100)
cetak_laporan(hasil_unit2)
cetak_ringkasan(hasil_unit2, energi_u2, jam_u2, 600)

Semua uji lulus

=== LAPORAN UNIT 1 ===
Hari  HR kcal/kWh   Efi %   CF %  SCC t/MWh  Status
   1        2,366   36.35   70.4      0.538  NORMAL
   2        2,351   36.58   77.0      0.534  NORMAL
   3        2,353   36.55   76.7      0.535  NORMAL
   4        2,356   36.51   77.0      0.535  NORMAL
   5        2,354   36.54   76.7      0.535  NORMAL
   6        2,350   36.59   76.7      0.534  NORMAL
   7        2,367   36.34   70.3      0.538  NORMAL
   8        2,364   36.38   70.1      0.537  NORMAL
   9        2,350   36.60   77.0      0.534  NORMAL
  10        2,357   36.49   76.8      0.536  NORMAL
  11        2,348   36.63   77.0      0.534  NORMAL
  12        2,348   36.63   76.9      0.534  NORMAL
  13        2,353   36.55   76.6      0.535  NORMAL
  14        2,383   36.10   69.9      0.541  EFISIENSI RENDAH
  15        2,379   36.15   70.3      0.541  EFISIENSI RENDAH
  16        2,350   36.59   77.0      0.534  NORMAL
  17        2,389   36.00   60.5      0.543  TIDAK PENUH